In [ ]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_products.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [ ]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))

EXPECTED_CURRENCY = {
    "cz": "CZK",
    "sk": "EUR",
    "hu": "HUF"
}[Path(SOURCE).stem.split("_")[-1]]

In [ ]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT sku) AS duplicate_skus,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(sku), '') IS NULL
               OR NULLIF(TRIM(product_name), '') IS NULL
               OR NULLIF(TRIM(category), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE is_bundle = 'FALSE'
              AND (
                  TRY_CAST(unit_cost_czk AS DECIMAL(18,2)) IS NULL
                  OR TRY_CAST(unit_cost_czk AS DECIMAL(18,2)) < 0
              )
        ) AS invalid_product_cost,

        COUNT(*) FILTER (
            WHERE is_bundle = 'TRUE'
        ) AS bundle_products,

        COUNT(*) FILTER (
            WHERE is_bundle = 'TRUE'
              AND NULLIF(TRIM(bundle_components), '') IS NULL
        ) AS missing_bundle_components

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

In [ ]:
# 03 | Bundle Investigation

show_sql(f"""
    SELECT
        sku,
        product_name,
        unit_cost_czk,
        bundle_components
    FROM read_csv('{SOURCE}', all_varchar = true)
    WHERE is_bundle = 'TRUE'
""")

In [ ]:
# 04 | Bundle Component Validation

show_sql(f"""
    WITH products AS (
        SELECT *
        FROM read_csv('{SOURCE}', all_varchar = true)
    ),
    components AS (
        SELECT
            p.sku AS bundle_sku,
            SPLIT_PART(component, ':', 1) AS component_sku
        FROM products p,
        UNNEST(STRING_SPLIT(p.bundle_components, '|')) AS t(component)
        WHERE p.is_bundle = 'TRUE'
    )
    SELECT
        COUNT(*) AS total_components,
        COUNT(*) FILTER (
            WHERE p.sku IS NULL
        ) AS missing_components,
        COUNT(*) FILTER (
            WHERE TRY_CAST(p.unit_cost_czk AS DECIMAL(18,2)) IS NULL
        ) AS missing_component_costs
    FROM components c
    LEFT JOIN products p ON c.component_sku = p.sku
""")

### 05 | Findings & Staging Recommendations – Products

| Area | Finding | Recommendation |
|---|---|---|
| Uniqueness | 28 products, no duplicate SKUs | Preserve SKU as the product key |
| Missing values | 4 bundles without unit costs | Derive bundle costs from components |
| Data validity | No missing required fields or invalid standalone product costs | Apply appropriate data types |
| Bundles | 4 bundles containing 12 component entries | Expand bundles in the business layer |
| Referential integrity | All 12 component entries have valid SKUs and costs | Use component costs for revenue allocation |

**Pending check:** Validate ordered SKUs against the product catalog.